# Student Notebook - MCP + Airbnb (Colab)

Reference notebook: local notes MCP + Airbnb MCP + optional real LLM.

## Install
Run once. npm only needed for the real Airbnb server.

In [ ]:
!pip install -q "mcp[cli]" nest_asyncio requests
!pip install -q azure-ai-inference

# Optional: real Airbnb server (only needed if USE_REAL_AIRBNB = True)
!npm install -g @openbnb/mcp-server-airbnb


In [ ]:
import sys
from ipykernel.iostream import OutStream

def _patched_fileno(self):
    # stdout → 1, stderr → 2
    if self is sys.stderr:
        return 2
    return 1

# Patch the class for all OutStream instances
OutStream.fileno = _patched_fileno

# And patch the current instances explicitly
sys.stdout.fileno = lambda: 1
sys.stderr.fileno = lambda: 2


## Config
Flip toggles as needed. Keep defaults for stubbed run.

In [ ]:

import os
from pathlib import Path

MCP_HTTP_TOKEN = os.getenv("MCP_HTTP_TOKEN", "devtoken123")
USE_REAL_AIRBNB = False  # Flip to True if you have npm and started `npx @openbnb/mcp-server-airbnb`
USE_REAL_LLM = False     # Flip to True if you have a GITHUB_TOKEN configured (see next cell)


In [ ]:
import os
BASE_ENV = os.environ.copy()
BASE_ENV["MCP_HTTP_TOKEN"] = MCP_HTTP_TOKEN


In [ ]:
import os

# This cell is only needed if USE_REAL_LLM = True. It is wrapped defensively so the
# notebook still runs end-to-end in stub mode even outside Colab or without the secret set.
try:
    from google.colab import userdata  # Colab secrets API

    # If your secret is saved under the key "GITHUB_TOKEN" in Colab:
    token = userdata.get("GITHUB_TOKEN")

    # If you used a different key name in the secrets UI, e.g. "github_token":
    # token = userdata.get("github_token")

    if token:
        os.environ["GITHUB_TOKEN"] = token
except Exception as e:
    print(f"Skipping Colab secret lookup ({e}); GITHUB_TOKEN only needed if USE_REAL_LLM = True.")

print("GITHUB_TOKEN visible to Python:", bool(os.getenv("GITHUB_TOKEN")))


## Local notes MCP server

In [ ]:

LOCAL_SERVER = Path("local_notes_server.py")
LOCAL_SERVER.write_text(
'''from mcp.server.fastmcp import FastMCP
notes = []
mcp = FastMCP("Notes")

@mcp.tool()
def add_note(text: str) -> str:
    'Add a note to the in-memory list.'
    notes.append(text)
    return f"Saved note #{len(notes)}: {text}"

@mcp.tool()
def list_notes() -> str:
    'List saved notes.'
    if not notes:
        return "No notes yet"
    return "\\n".join(f"{i+1}. {n}" for i, n in enumerate(notes))

if __name__ == "__main__":
    mcp.run()
'''.strip() + "",
    encoding="utf-8",
)
print("wrote", LOCAL_SERVER)


## Fake Airbnb MCP server (fallback when `USE_REAL_AIRBNB = False`)
Returns a few fixed, made-up listings per city so the whole pipeline runs without npm/npx.

In [ ]:

FAKE_AIRBNB_SERVER = Path("fake_airbnb_server.py")
FAKE_AIRBNB_SERVER.write_text(
'''from mcp.server.fastmcp import FastMCP

mcp = FastMCP("FakeAirbnb")

FAKE_LISTINGS = {
    "paris": [
        {"name": "Cozy studio near the Eiffel Tower", "price_per_night": 95, "rating": 4.8},
        {"name": "Le Marais loft with balcony", "price_per_night": 120, "rating": 4.6},
    ],
    "new york": [
        {"name": "Brooklyn brownstone room", "price_per_night": 110, "rating": 4.7},
        {"name": "Manhattan studio near Central Park", "price_per_night": 180, "rating": 4.5},
    ],
    "london": [
        {"name": "Notting Hill flat", "price_per_night": 130, "rating": 4.9},
        {"name": "Shoreditch loft", "price_per_night": 105, "rating": 4.4},
    ],
}

@mcp.tool()
def search_listings(location: str) -> str:
    "Return a few fixed, made-up Airbnb-style listings for the given city."
    key = location.strip().lower()
    listings = FAKE_LISTINGS.get(key)
    if not listings:
        return f"No stub listings for {location}. Try one of: {', '.join(FAKE_LISTINGS)}"
    lines = [f"{item['name']} - ${item['price_per_night']}/night - {item['rating']} stars" for item in listings]
    return "\\n".join(lines)

if __name__ == "__main__":
    mcp.run()
'''.strip() + "",
    encoding="utf-8",
)
print("wrote", FAKE_AIRBNB_SERVER)


## Client helpers (convert tools, stub planner, optional real LLM)

In [ ]:

import asyncio
import json
import re
import nest_asyncio
from typing import Any, Dict, List
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()

def convert_tool(tool, prefix: str):
    # Azure requires ^[a-zA-Z0-9_\.-]+$, so no slashes
    fn_name = f"{prefix}__{tool.name}"
    return {
        "type": "function",
        "function": {
            "name": fn_name,
            "description": tool.description or "mcp tool",
            "parameters": {
                "type": "object",
                "properties": tool.inputSchema.get("properties", {}),
                "required": tool.inputSchema.get("required", []),
            },
        },
    }


def stub_plan(prompt: str, functions: List[Dict[str, Any]]):
    """Minimal rule-based planner used when USE_REAL_LLM is False. Looks at the
    prefixed tool names (notes__..., airbnb__...) and simple keywords in the
    prompt to decide which tool(s), if any, to call."""
    prompt_lower = prompt.lower()
    by_name = {f["function"]["name"]: f["function"] for f in functions}
    calls = []

    # Airbnb-style search: trigger on "listing"/"airbnb"/"stay"/etc., extract a
    # capitalized city name after "in" (e.g. "listing in Paris" -> "Paris").
    airbnb_fn = next(
        (n for n in by_name if n.startswith("airbnb__") and ("search" in n or "listing" in n)),
        None,
    )
    if airbnb_fn and any(k in prompt_lower for k in ["airbnb", "listing", "stay", "rent", "book"]):
        m = re.search(r"in ([A-Z][a-zA-Z]+(?:\s[A-Z][a-zA-Z]+)?)", prompt)
        city = m.group(1) if m else "Paris"
        props = by_name[airbnb_fn]["parameters"].get("properties", {})
        args = {}
        for key in props:
            if any(k in key.lower() for k in ["location", "city", "place"]):
                args[key] = city
        if not args and props:
            args = {next(iter(props)): city}
        calls.append({"name": airbnb_fn, "args": args})

    # Notes: save a note if the user says "note"/"remember"/"save".
    add_note_fn = next((n for n in by_name if n.startswith("notes__add_note")), None)
    if add_note_fn and any(k in prompt_lower for k in ["note", "remember", "save this"]):
        calls.append({"name": add_note_fn, "args": {"text": prompt}})

    # Notes: list notes if the user asks to see them.
    list_notes_fn = next((n for n in by_name if n.startswith("notes__list_notes")), None)
    if list_notes_fn and any(k in prompt_lower for k in ["list my notes", "show my notes", "list notes", "what notes"]):
        calls.append({"name": list_notes_fn, "args": {}})

    return calls


def call_llm(prompt: str, functions: List[Dict[str, Any]], use_real: bool = False):
    if not use_real:
        return stub_plan(prompt, functions)

    import os
    from azure.ai.inference import ChatCompletionsClient
    from azure.core.credentials import AzureKeyCredential
    token = os.getenv("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Set GITHUB_TOKEN or use stub planner.")
    client = ChatCompletionsClient("https://models.inference.ai.azure.com", AzureKeyCredential(token))
    resp = client.complete(
        model="gpt-4o",
        messages=[
            {
                "role": "system",
                "content": "Plan MCP tool calls. Call a tool only when it helps answer the user's request.",
            },
            {"role": "user", "content": prompt},
        ],
        tools=functions,
        temperature=0,
        max_tokens=400,
    )
    calls = []
    msg = resp.choices[0].message
    for tc in msg.tool_calls or []:
        args = tc.function.arguments
        args_json = json.loads(args) if isinstance(args, str) else args
        calls.append({"name": tc.function.name, "args": args_json})
    return calls


In [ ]:
def answer_with_llm(
    user_prompt: str,
    tool_calls: List[Dict[str, Any]],
    tool_results: List[Dict[str, Any]],
    use_real: bool = False,
) -> str:
    import os
    import json

    if not use_real:
        # Stub summary: no LLM call, just render the tool results directly.
        if not tool_results:
            return "No tools were called for this prompt.\n\n## Tools used\n- None"
        lines = ["Here's what I found:\n"]
        for r in tool_results:
            content = r.get("content", [])
            snippet = content[0] if content else "(no content)"
            lines.append(f"- **{r.get('name')}** (args={r.get('args')}):\n{snippet}\n")
        lines.append("## Tools used")
        for name in sorted({r.get("name") for r in tool_results}):
            lines.append(f"- {name}")
        return "\n".join(lines)

    # MINIMAL FIX: shrink tool_results before sending to gpt-4o
    small_results = []
    for r in tool_results:
        content = r.get("content", [])
        short_content = []
        if content:
            first = content[0]
            if isinstance(first, str) and len(first) > 4000:
                first = first[:4000] + "...(truncated)..."
            short_content = [first]
        small_results.append(
            {
                "name": r.get("name"),
                "args": r.get("args", {}),
                "content": short_content,
            }
        )


    from azure.ai.inference import ChatCompletionsClient
    from azure.core.credentials import AzureKeyCredential

    token = os.getenv("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Set GITHUB_TOKEN or use_real=False in answer_with_llm.")

    client = ChatCompletionsClient(
        "https://models.inference.ai.azure.com",
        AzureKeyCredential(token),
    )

    payload = {
        "user_question": user_prompt,
        "tool_calls": tool_calls,
        # use the shrunk version here
        "tool_results": small_results,
    }

    resp = client.complete(
        model="gpt-4o",
        messages=[
            {
                "role": "system",
                "content": (
                    "You answer the user's question using the given tool outputs.\n"
                    "JSON contains user_question, tool_calls, and tool_results (already truncated).\n"
                    "1. Answer clearly in markdown.\n"
                    "2. At the end, add:\n"
                    "## Tools used\n"
                    "- One bullet per distinct tool name.\n"
                ),
            },
            {
                "role": "user",
                "content": json.dumps(payload, ensure_ascii=False),
            },
        ],
        temperature=0,
        max_tokens=600,
    )

    msg = resp.choices[0].message
    parts = getattr(msg, "content", None)
    if isinstance(parts, list):
        texts = []
        for p in parts:
            text = getattr(p, "text", None) or getattr(p, "content", None)
            if isinstance(text, str):
                texts.append(text)
        if texts:
            return "".join(texts)

    return str(msg.content)


## Orchestrate (connect both servers and execute tool_calls)

In [ ]:
import sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def orchestrate(prompt: str):
    local_params = StdioServerParameters(
        command="mcp",
        args=["run", str(LOCAL_SERVER)],
        env=BASE_ENV,  # <- use merged env
    )

    if USE_REAL_AIRBNB:
        airbnb_params = StdioServerParameters(
            command="npx",
            args=["@openbnb/mcp-server-airbnb", "--ignore-robots-txt"],
            env=BASE_ENV,
        )
    else:
        # Fallback: spawn the fake Airbnb MCP server with this same Python
        # interpreter instead of relying on npm/npx being installed.
        airbnb_params = StdioServerParameters(
            command=sys.executable,
            args=[str(FAKE_AIRBNB_SERVER)],
            env=BASE_ENV,
        )

    async with stdio_client(local_params) as (lr, lw):
        async with ClientSession(lr, lw) as local_sess:
            await local_sess.initialize()
            local_tools = await local_sess.list_tools()

            async with stdio_client(airbnb_params) as (ar, aw):
                async with ClientSession(ar, aw) as airbnb_sess:
                    await airbnb_sess.initialize()
                    airbnb_tools = await airbnb_sess.list_tools()

                    functions = (
                        [convert_tool(t, "notes") for t in local_tools.tools]
                        + [convert_tool(t, "airbnb") for t in airbnb_tools.tools]
                    )

                    tool_calls = call_llm(prompt, functions, use_real=USE_REAL_LLM)
                    print("tool_calls:", tool_calls)

                    tool_results = []
                    for call in tool_calls:
                        name = call["name"]
                        args = call["args"]
                        prefix, tool_name = name.split("__", 1)

                        if prefix == "notes":
                            res = await local_sess.call_tool(tool_name, args)
                            tool_results.append(
                                {
                                    "name": name,
                                    "args": args,
                                    "content": [c.text for c in res.content if hasattr(c, "text")],
                                }
                            )
                        elif prefix == "airbnb":
                            res = await airbnb_sess.call_tool(tool_name, args)
                            payload = []
                            if hasattr(res, "content"):
                                for c in res.content:
                                    if hasattr(c, "text"):
                                        payload.append(c.text)
                            tool_results.append(
                                {
                                    "name": name,
                                    "args": args,
                                    "content": payload,
                                }
                            )

                    # note: DO NOT call answer_with_llm here if you want to inspect things first
                    return tool_calls, tool_results

## Demo
Adjust the prompt as you like. Switch `USE_REAL_AIRBNB/USE_REAL_LLM` to true when ready.

In [ ]:
# Task 2: try a prompt that should trigger BOTH the notes tool and the Airbnb
# (real or fake) tool, so we can confirm the full mini-agent pipeline end to end.
prompt = "Find me an Airbnb listing in Paris and save a note that I'm looking at Paris trips."

tool_calls, tool_results = await orchestrate(prompt)

print("\n=== tool_calls ===")
print(tool_calls)

print("\n=== tool_results ===")
for r in tool_results:
    print(f"- {r['name']}(args={r['args']}) ->")
    for line in r["content"]:
        print("   ", line)

print("\n=== final answer (answer_with_llm, use_real=USE_REAL_LLM) ===")
summary = answer_with_llm(prompt, tool_calls, tool_results, use_real=USE_REAL_LLM)
print(summary)
